# Spatial Data Preparation and Travel Graph Construction

This notebook constructs the **whole-Iligan** layered travel graph for the thesis.

It follows the clarified sequence:
1. extract the whole Iligan walk and drive graphs
2. save `nodes_uncategorized.csv`
3. save `nodes_walk.csv`
4. save `nodes_ride.csv` where **ride = drivable node layer**
5. build bidirectional start-walk edges
6. build bidirectional end-walk edges
7. build bidirectional ride/drivable edges
8. build **wait**
9. build **alight**
10. build **direct**
11. build **transfer**
12. compute `accessible_nodes` for every edge *(new)*
13. stitch the layered graph
14. generate **zoomable HTML previews** for verification
15. define `JeepneyRoute` and instantiate `TravelGraphManager` *(new)*

## 0. Optional installation cell

In [64]:
# Uncomment and run only if needed.
%pip install osmnx geopandas folium networkx shapely pandas numpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [65]:

from __future__ import annotations

from pathlib import Path
from IPython.display import IFrame, display

import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
from shapely.geometry import LineString


## 1. Project configuration

In [66]:

# -----------------------------
# Study-area settings
# -----------------------------
PLACE_QUERY_CANDIDATES = [
    "Iligan City, Philippines"
]
POINT_FALLBACK_QUERY = "Iligan City, Philippines"
POINT_FALLBACK_DIST_M = 30000

# -----------------------------
# Matching and graph settings
# -----------------------------
COORD_ROUNDING = 7
MAX_INTERLAYER_SNAP_DIST_M = 80.0
SIMPLIFY_GRAPHS = True
RETAIN_ALL_COMPONENTS = True

# -----------------------------
# Project folders
# -----------------------------
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PREVIEW_DIR = OUTPUT_DIR / "previews"

for folder in [DATA_DIR, OUTPUT_DIR, PREVIEW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Output file paths
# -----------------------------
NODES_UNCATEGORIZED_CSV = OUTPUT_DIR / "nodes_uncategorized.csv"
NODES_WALK_CSV = OUTPUT_DIR / "nodes_walk.csv"
NODES_RIDE_CSV = OUTPUT_DIR / "nodes_ride.csv"

EDGES_START_WALK_CSV = OUTPUT_DIR / "edges_start_walk.csv"
EDGES_END_WALK_CSV = OUTPUT_DIR / "edges_end_walk.csv"
EDGES_RIDE_CSV = OUTPUT_DIR / "edges_ride.csv"
EDGES_WAIT_CSV = OUTPUT_DIR / "edges_wait.csv"
EDGES_ALIGHT_CSV = OUTPUT_DIR / "edges_alight.csv"
EDGES_DIRECT_CSV = OUTPUT_DIR / "edges_direct.csv"
EDGES_TRANSFER_CSV = OUTPUT_DIR / "edges_transfer.csv"

TRAVEL_GRAPH_NODES_CSV = OUTPUT_DIR / "travel_graph_nodes.csv"
TRAVEL_GRAPH_EDGES_CSV = OUTPUT_DIR / "iligan_travel_graph.csv"

PREVIEW_WALK_NODES_HTML = PREVIEW_DIR / "preview_walk_nodes.html"
PREVIEW_RIDE_NODES_HTML = PREVIEW_DIR / "preview_ride_nodes.html"
PREVIEW_RIDE_NETWORK_HTML = PREVIEW_DIR / "preview_ride_network.html"
PREVIEW_STITCHED_HTML = PREVIEW_DIR / "preview_travel_graph_stitched.html"

ox.settings.use_cache = True
ox.settings.log_console = True


## 2. Helper functions

In [67]:

def make_coord_key(df: pd.DataFrame, lon_col: str = "lon", lat_col: str = "lat", decimals: int = 7) -> pd.Series:
    return df[lon_col].round(decimals).astype(str) + "|" + df[lat_col].round(decimals).astype(str)


def resolve_study_area_boundary(place_queries: list):
    last_error = None
    for query in place_queries:
        try:
            gdf = ox.geocode_to_gdf(query)
            if gdf.empty:
                continue

            chosen = gdf.copy()
            if {"class", "type"}.issubset(chosen.columns):
                mask = (
                    chosen["class"].astype(str).str.lower().eq("boundary")
                    & chosen["type"].astype(str).str.lower().eq("administrative")
                )
                if mask.any():
                    chosen = chosen.loc[mask].copy()

            chosen = chosen.iloc[[0]].copy()
            geom = chosen.geometry.iloc[0]
            if geom is None or geom.is_empty:
                continue

            return chosen, f"geocode_to_gdf({query!r})"
        except Exception as exc:
            last_error = exc

    if last_error is not None:
        raise RuntimeError(f"Unable to resolve the Iligan boundary: {last_error}")
    raise RuntimeError("Unable to resolve the Iligan boundary from the configured place queries.")


def load_graphs_for_study_area(
    place_queries: list,
    point_query: str | None = None,
    point_dist: float = 30000,
    simplify: bool = True,
    retain_all: bool = True,
):
    study_area_gdf, boundary_source = resolve_study_area_boundary(place_queries)
    polygon = study_area_gdf.geometry.union_all() if hasattr(study_area_gdf.geometry, "union_all") else study_area_gdf.unary_union

    try:
        G_walk_raw = ox.graph_from_polygon(
            polygon,
            network_type="walk",
            simplify=simplify,
            retain_all=retain_all,
        )
        G_drive_raw = ox.graph_from_polygon(
            polygon,
            network_type="drive",
            simplify=simplify,
            retain_all=retain_all,
        )
        graph_source = "graph_from_polygon(study_area_boundary)"
    except Exception as exc:
        if point_query is None:
            raise
        point = ox.geocode(point_query)
        G_walk_raw = ox.graph_from_point(
            point,
            dist=point_dist,
            network_type="walk",
            simplify=simplify,
            retain_all=retain_all,
        )
        G_drive_raw = ox.graph_from_point(
            point,
            dist=point_dist,
            network_type="drive",
            simplify=simplify,
            retain_all=retain_all,
        )
        graph_source = f"graph_from_point({point_query!r}, dist={point_dist}) because polygon download failed: {type(exc).__name__}: {exc}"

    G_walk_proj = ox.project_graph(G_walk_raw)
    G_drive_proj = ox.project_graph(G_drive_raw)
    return study_area_gdf, boundary_source, graph_source, G_walk_raw, G_walk_proj, G_drive_raw, G_drive_proj


def node_table_from_graph(G_raw: nx.MultiDiGraph, G_proj: nx.MultiDiGraph) -> pd.DataFrame:
    raw_nodes = pd.DataFrame.from_dict(dict(G_raw.nodes(data=True)), orient="index").reset_index()
    raw_nodes = raw_nodes.rename(columns={"index": "base_node_id", "x": "lon", "y": "lat"})

    proj_nodes = pd.DataFrame.from_dict(dict(G_proj.nodes(data=True)), orient="index").reset_index()
    proj_nodes = proj_nodes.rename(columns={"index": "base_node_id", "x": "x", "y": "y"})

    merged = proj_nodes[["base_node_id", "x", "y"]].merge(
        raw_nodes[["base_node_id", "lon", "lat"]],
        on="base_node_id",
        how="left",
    )
    merged["coord_key"] = make_coord_key(merged, "lon", "lat", COORD_ROUNDING)
    merged["node_id"] = merged["base_node_id"].astype(str)
    return merged[["node_id", "base_node_id", "x", "y", "lon", "lat", "coord_key"]].sort_values("base_node_id").reset_index(drop=True)


def extract_uncategorized_nodes(walk_nodes: pd.DataFrame, drive_nodes: pd.DataFrame) -> pd.DataFrame:
    walk = walk_nodes.copy()
    walk["in_walk_graph"] = True
    walk["in_drive_graph"] = False

    drive = drive_nodes.copy()
    drive["in_walk_graph"] = False
    drive["in_drive_graph"] = True

    union = pd.concat([walk, drive], ignore_index=True, sort=False)
    union = union.groupby("base_node_id", as_index=False).agg(
        {
            "node_id": "first",
            "x": "first",
            "y": "first",
            "lon": "first",
            "lat": "first",
            "coord_key": "first",
            "in_walk_graph": "max",
            "in_drive_graph": "max",
        }
    )
    union["node_id"] = union["base_node_id"].astype(str)
    return union[
        ["node_id", "base_node_id", "x", "y", "lon", "lat", "in_walk_graph", "in_drive_graph", "coord_key"]
    ].sort_values("base_node_id").reset_index(drop=True)


def graph_edges_to_bidirectional_base(G_proj: nx.MultiDiGraph, prefix: str, edge_type: str) -> pd.DataFrame:
    edges_gdf = ox.graph_to_gdfs(G_proj, nodes=False, edges=True).reset_index()

    if "length" not in edges_gdf.columns:
        edges_gdf["length"] = edges_gdf.geometry.length

    edges_gdf["pair_key"] = edges_gdf.apply(lambda row: tuple(sorted((row["u"], row["v"]))), axis=1)
    edges_gdf = (
        edges_gdf.sort_values(["pair_key", "length"])
        .drop_duplicates(subset=["pair_key"], keep="first")
        .copy()
    )

    rows = []
    for row in edges_gdf.itertuples(index=False):
        pair_u, pair_v = row.pair_key
        dist = float(row.length)
        geom = row.geometry

        rows.append(
            {
                "u": f"{prefix}_{pair_u}",
                "v": f"{prefix}_{pair_v}",
                "dist": dist,
                "edge_type": edge_type,
                "geometry": geom,
            }
        )
        rev_geom = LineString(list(geom.coords)[::-1]) if geom is not None else None
        rows.append(
            {
                "u": f"{prefix}_{pair_v}",
                "v": f"{prefix}_{pair_u}",
                "dist": dist,
                "edge_type": edge_type,
                "geometry": rev_geom,
            }
        )
    return pd.DataFrame(rows)


def duplicate_walk_nodes_to_layers(nodes_walk: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    start_nodes = nodes_walk.copy()
    start_nodes["node_id"] = "sw_" + start_nodes["base_node_id"].astype(str)

    end_nodes = nodes_walk.copy()
    end_nodes["node_id"] = "ew_" + end_nodes["base_node_id"].astype(str)

    return start_nodes, end_nodes


def build_direct_edges(start_nodes: pd.DataFrame, end_nodes: pd.DataFrame) -> pd.DataFrame:
    merged = start_nodes[["base_node_id", "node_id", "x", "y"]].merge(
        end_nodes[["base_node_id", "node_id", "x", "y"]],
        on="base_node_id",
        how="inner",
        suffixes=("_start", "_end"),
    )

    rows = []
    for row in merged.itertuples(index=False):
        rows.append(
            {
                "u": row.node_id_start,
                "v": row.node_id_end,
                "dist": 0.0,
                "edge_type": "direct",
                "geometry": LineString([(row.x_start, row.y_start), (row.x_end, row.y_end)]),
            }
        )
    return pd.DataFrame(rows)


def _prepare_match_df(df: pd.DataFrame, node_col: str) -> pd.DataFrame:
    return df[[node_col, "base_node_id", "x", "y", "lon", "lat", "coord_key"]].drop_duplicates(node_col).reset_index(drop=True)


def build_interlayer_edges(
    left_df: pd.DataFrame,
    right_df: pd.DataFrame,
    left_node_col: str,
    right_node_col: str,
    edge_type: str,
    max_snap_dist_m: float | None = None,
    anchor_side: str = "right",
) -> pd.DataFrame:
    if left_df.empty or right_df.empty:
        return pd.DataFrame(columns=["u", "v", "dist", "edge_type", "geometry"])

    if anchor_side not in {"left", "right"}:
        raise ValueError("anchor_side must be 'left' or 'right'.")

    source_df = left_df if anchor_side == "left" else right_df
    target_df = right_df if anchor_side == "left" else left_df
    source_node_col = left_node_col if anchor_side == "left" else right_node_col
    target_node_col = right_node_col if anchor_side == "left" else left_node_col

    source = _prepare_match_df(source_df, source_node_col)
    target = _prepare_match_df(target_df, target_node_col)

    records = []

    exact = source.merge(target, on="coord_key", how="inner", suffixes=("_source", "_target"))
    source_id_key = source_node_col + "_source" if source_node_col + "_source" in exact.columns else source_node_col
    target_id_key = target_node_col + "_target" if target_node_col + "_target" in exact.columns else target_node_col

    exact_source_ids = set()
    for rec in exact.to_dict("records"):
        exact_source_ids.add(rec[source_id_key])
        records.append(
            {
                "source_node": rec[source_id_key],
                "target_node": rec[target_id_key],
                "x_source": rec["x_source"],
                "y_source": rec["y_source"],
                "x_target": rec["x_target"],
                "y_target": rec["y_target"],
            }
        )

    source_remaining = source[~source[source_node_col].isin(exact_source_ids)].copy()
    if not source_remaining.empty and not target.empty:
        target_xy = target[["x", "y"]].to_numpy(dtype=float)
        for rec in source_remaining.to_dict("records"):
            sx = float(rec["x"])
            sy = float(rec["y"])
            dists = np.sqrt((target_xy[:, 0] - sx) ** 2 + (target_xy[:, 1] - sy) ** 2)
            nearest_idx = int(dists.argmin())
            snap_dist = float(dists[nearest_idx])

            if max_snap_dist_m is not None and snap_dist > float(max_snap_dist_m):
                continue

            target_row = target.iloc[nearest_idx].to_dict()
            records.append(
                {
                    "source_node": rec[source_node_col],
                    "target_node": target_row[target_node_col],
                    "x_source": rec["x"],
                    "y_source": rec["y"],
                    "x_target": target_row["x"],
                    "y_target": target_row["y"],
                }
            )

    rows = []
    for rec in records:
        if anchor_side == "left":
            u = rec["source_node"]
            v = rec["target_node"]
            x_u, y_u = rec["x_source"], rec["y_source"]
            x_v, y_v = rec["x_target"], rec["y_target"]
        else:
            u = rec["target_node"]
            v = rec["source_node"]
            x_u, y_u = rec["x_target"], rec["y_target"]
            x_v, y_v = rec["x_source"], rec["y_source"]

        rows.append(
            {
                "u": u,
                "v": v,
                "dist": 0.0,
                "edge_type": edge_type,
                "geometry": LineString([(x_u, y_u), (x_v, y_v)]),
            }
        )
    return pd.DataFrame(rows)


def assign_edge_ids(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    out = df.copy().reset_index(drop=True)
    out.insert(0, "edge_id", [f"{prefix}_{i + 1:06d}" for i in range(len(out))])
    return out


# CHANGED: added 'accessible_nodes' to save_cols so the column is written
# to every per-type CSV and the master stitched CSV after Section 19.5 runs.
def edge_frame_to_csv(df: pd.DataFrame, csv_path: Path) -> None:
    out = df.copy()
    save_cols = [col for col in ["edge_id", "u", "v", "dist", "edge_type", "accessible_nodes"] if col in out.columns]
    out = out[save_cols]
    out.to_csv(csv_path, index=False)


def node_frame_to_csv(df: pd.DataFrame, csv_path: Path, keep_cols: list[str]) -> None:
    out = df.copy()
    out = out[keep_cols]
    out.to_csv(csv_path, index=False)


def make_edges_gdf(df: pd.DataFrame, crs):
    import geopandas as gpd
    out = df.copy()
    if "geometry" not in out.columns:
        out["geometry"] = pd.Series(dtype="object")
    return gpd.GeoDataFrame(out, geometry="geometry", crs=crs)


def make_nodes_gdf(df: pd.DataFrame, crs):
    import geopandas as gpd
    gdf = gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df["x"], df["y"]),
        crs=crs,
    )
    return gdf


def add_nodes_to_digraph(G: nx.DiGraph, nodes_df: pd.DataFrame) -> None:
    for row in nodes_df.itertuples(index=False):
        attrs = row._asdict()
        node_id = attrs.pop("node_id")
        attrs.pop("coord_key", None)
        G.add_node(node_id, **attrs)


def add_edges_to_digraph(G: nx.DiGraph, edges_df: pd.DataFrame) -> None:
    for row in edges_df.itertuples(index=False):
        attrs = row._asdict()
        edge_id = attrs.pop("edge_id")
        geom = attrs.pop("geometry", None)
        u = attrs.pop("u")
        v = attrs.pop("v")
        G.add_edge(u, v, edge_id=edge_id, geometry=geom, **attrs)


def add_gdf_to_folium(
    fmap,
    gdf,
    name: str,
    color: str = "#3388ff",
    weight: float = 2,
    opacity: float = 0.8,
    fill_opacity: float = 0.1,
    show: bool = True,
    popup_fields: list[str] | None = None,
    is_point_layer: bool = False,
):
    import folium

    if gdf is None or len(gdf) == 0:
        return

    gdf_ll = gdf.to_crs(4326) if str(gdf.crs) != "EPSG:4326" else gdf

    popup = None
    if popup_fields:
        valid_fields = [f for f in popup_fields if f in gdf_ll.columns]
        if valid_fields:
            popup = folium.GeoJsonPopup(fields=valid_fields, labels=True)

    style_kwargs = {
        "color": color,
        "weight": weight,
        "opacity": opacity,
        "fillOpacity": fill_opacity,
    }
    if is_point_layer:
        style_kwargs["radius"] = 2

    folium.GeoJson(
        gdf_ll,
        name=name,
        show=show,
        style_function=lambda _x, style_kwargs=style_kwargs: style_kwargs,
        marker=folium.CircleMarker(radius=2, color=color, fill=True, fill_opacity=0.8) if is_point_layer else None,
        popup=popup,
        zoom_on_click=False,
    ).add_to(fmap)


def create_interactive_map(
    boundary_gdf,
    output_html_path: Path,
    title: str,
    walk_nodes_gdf=None,
    ride_nodes_gdf=None,
    walk_edges_gdf=None,
    ride_edges_gdf=None,
    wait_edges_gdf=None,
    alight_edges_gdf=None,
    direct_edges_gdf=None,
    transfer_edges_gdf=None,
):
    import folium
    from folium.plugins import Fullscreen

    boundary_ll = boundary_gdf.to_crs(4326) if str(boundary_gdf.crs) != "EPSG:4326" else boundary_gdf
    centroid = boundary_ll.geometry.iloc[0].centroid

    fmap = folium.Map(
        location=[centroid.y, centroid.x],
        zoom_start=12,
        tiles="CartoDB positron",
        control_scale=True,
    )
    Fullscreen().add_to(fmap)

    title_html = f'''
         <div style="position: fixed;
                     top: 10px; left: 50px; width: 440px; z-index:9999;
                     background-color: white; padding: 10px; border: 1px solid #999;">
             <b>{title}</b>
         </div>
    '''
    fmap.get_root().html.add_child(folium.Element(title_html))

    add_gdf_to_folium(
        fmap,
        boundary_ll,
        name="Iligan boundary",
        color="#111111",
        weight=2.5,
        opacity=1.0,
        fill_opacity=0.02,
        show=True,
    )

    add_gdf_to_folium(
        fmap,
        walk_nodes_gdf,
        name="Walk nodes",
        color="#1f77b4",
        weight=1,
        opacity=0.9,
        fill_opacity=0.8,
        show=True,
        popup_fields=["node_id", "base_node_id", "lon", "lat"],
        is_point_layer=True,
    )
    add_gdf_to_folium(
        fmap,
        ride_nodes_gdf,
        name="Drivable nodes",
        color="#d62728",
        weight=1,
        opacity=0.9,
        fill_opacity=0.8,
        show=True,
        popup_fields=["node_id", "base_node_id", "lon", "lat"],
        is_point_layer=True,
    )
    add_gdf_to_folium(
        fmap,
        walk_edges_gdf,
        name="Walk edges",
        color="#7f7f7f",
        weight=1.2,
        opacity=0.55,
        show=False,
        popup_fields=["edge_id", "u", "v", "dist", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        ride_edges_gdf,
        name="Drivable edges",
        color="#d62728",
        weight=2.2,
        opacity=0.9,
        show=True,
        popup_fields=["edge_id", "u", "v", "dist", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        wait_edges_gdf,
        name="Wait edges",
        color="#2ca02c",
        weight=1.8,
        opacity=0.8,
        show=False,
        popup_fields=["edge_id", "u", "v", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        alight_edges_gdf,
        name="Alight edges",
        color="#9467bd",
        weight=1.8,
        opacity=0.8,
        show=False,
        popup_fields=["edge_id", "u", "v", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        direct_edges_gdf,
        name="Direct edges",
        color="#8c564b",
        weight=1.6,
        opacity=0.8,
        show=False,
        popup_fields=["edge_id", "u", "v", "edge_type"],
    )
    add_gdf_to_folium(
        fmap,
        transfer_edges_gdf,
        name="Transfer edges",
        color="#ff7f0e",
        weight=1.8,
        opacity=0.8,
        show=False,
        popup_fields=["edge_id", "u", "v", "edge_type"],
    )

    bounds = boundary_ll.total_bounds
    fmap.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    folium.LayerControl(collapsed=False).add_to(fmap)
    fmap.save(str(output_html_path))
    return fmap


def show_html_preview(path: Path, height: int = 720):
    if path.exists():
        display(IFrame(src=str(path), width="100%", height=height))
    else:
        print(f"Preview file does not exist yet: {path}")


## 2.5. `accessible_nodes` helper functions

**NEW** — these two functions compute and attach the `accessible_nodes` column.

**Design rationale:** `accessible_nodes` for an edge `(u→v)` is the semicolon-delimited
list of every `edge_id` whose `u` equals this edge's `v` in the full stitched graph.
The node prefixes (`sw_`, `ride_`, `ew_`) already enforce all valid layer transitions
structurally, so no additional rule-checking is required — the topology does it automatically.

In [68]:

def compute_v_to_outgoing(all_edges: pd.DataFrame) -> dict:
    """
    Build {node_id -> [edge_id, ...]} mapping for all edges departing FROM that node.

    Must be called on the FULLY STITCHED travel_graph_edges DataFrame so that
    cross-layer connections (wait, alight, transfer, direct) are included.
    """
    return (
        all_edges.groupby("u", sort=False)["edge_id"]
        .apply(list)
        .to_dict()
    )


def attach_accessible_nodes(edges_df: pd.DataFrame, v_to_outgoing: dict) -> pd.DataFrame:
    """
    Add the accessible_nodes column to an edge DataFrame.

    For each row, accessible_nodes is a semicolon-delimited string of all
    edge_ids reachable from that edge's v in the full graph.
    Terminal nodes (no onward edges) receive an empty string.
    """
    out = edges_df.copy()
    out["accessible_nodes"] = out["v"].apply(
        lambda v_node: ";".join(v_to_outgoing.get(v_node, []))
    )
    return out


## 2.6. `JeepneyRoute` class

**NEW** — wraps a single jeepney route as an ordered, circular sequence of `ride_` node IDs.

The ride graph is a *reference* for where jeeps *may* travel. A `JeepneyRoute` defines
which of those roads a specific jeepney *actually* uses. Only routes supplied to
`TravelGraphManager` are accessible to the passenger; the rest of the ride layer is
invisible to routing.

In [69]:

class JeepneyRoute:
    """
    A single jeepney route: an ordered circular sequence of ride-layer node IDs.

    Parameters
    ----------
    route_id : str
        Unique identifier (e.g. "R1", "Iligan-Hinaplanon").
    nodes : list[str]
        Ordered ride_ node IDs. The route is circular — the last node connects
        back to the first. Do NOT repeat the first node at the end.

    Example
    -------
    >>> r = JeepneyRoute("R1", ["ride_A", "ride_B", "ride_C"])
    # circular edges: ride_A→ride_B, ride_B→ride_C, ride_C→ride_A
    """

    def __init__(self, route_id: str, nodes: list) -> None:
        if len(nodes) < 2:
            raise ValueError(f"Route {route_id!r} needs at least 2 nodes.")
        bad = [n for n in nodes if not str(n).startswith("ride_")]
        if bad:
            raise ValueError(
                f"Route {route_id!r} has non-ride nodes: {bad[:5]}. "
                "All nodes must carry the 'ride_' prefix."
            )
        self._route_id = str(route_id)
        self._nodes    = list(nodes)

    # ── properties ──────────────────────────────────────────────────────────

    @property
    def route_id(self) -> str:
        """Unique route identifier."""
        return self._route_id

    @property
    def nodes(self) -> list:
        """Ordered ride node IDs (first node not repeated at end)."""
        return list(self._nodes)

    @property
    def edge_pairs(self) -> set:
        """
        Set of directed (u, v) ride-edge pairs that belong to this route,
        including the circular wrap-around from the last node to the first.
        """
        n = self._nodes
        pairs = set(zip(n, n[1:]))   # consecutive pairs
        pairs.add((n[-1], n[0]))      # close the loop
        return pairs

    @property
    def node_set(self) -> set:
        """Set of all ride node IDs on this route."""
        return set(self._nodes)

    def __len__(self) -> int:
        return len(self._nodes)

    def __repr__(self) -> str:
        return f"JeepneyRoute(id={self._route_id!r}, stops={len(self._nodes)})"


## 3. Extract the whole Iligan walk and drive graphs

In [70]:

study_area_gdf, boundary_source, graph_source, G_walk_raw, G_walk_proj, G_drive_raw, G_drive_proj = load_graphs_for_study_area(
    place_queries=PLACE_QUERY_CANDIDATES,
    point_query=POINT_FALLBACK_QUERY,
    point_dist=POINT_FALLBACK_DIST_M,
    simplify=SIMPLIFY_GRAPHS,
    retain_all=RETAIN_ALL_COMPONENTS,
)

print("Boundary source:", boundary_source)
print("Graph source:", graph_source)
print("Walk graph:", len(G_walk_proj.nodes), "nodes |", len(G_walk_proj.edges), "edges")
print("Drive graph:", len(G_drive_proj.nodes), "nodes |", len(G_drive_proj.edges), "edges")
print("Projected CRS:", G_walk_proj.graph.get("crs"))


Boundary source: geocode_to_gdf('Iligan City, Philippines')
Graph source: graph_from_polygon(study_area_boundary)
Walk graph: 4689 nodes | 12014 edges
Drive graph: 3477 nodes | 8851 edges
Projected CRS: EPSG:32651


## 4. Extract raw nodes plus attributes and save `nodes_uncategorized.csv`

In [71]:

walk_nodes_lookup = node_table_from_graph(G_walk_raw, G_walk_proj).copy()
drive_nodes_lookup = node_table_from_graph(G_drive_raw, G_drive_proj).copy()

nodes_uncategorized = extract_uncategorized_nodes(walk_nodes_lookup, drive_nodes_lookup)
node_frame_to_csv(
    nodes_uncategorized,
    NODES_UNCATEGORIZED_CSV,
    keep_cols=["node_id", "base_node_id", "x", "y", "lon", "lat", "in_walk_graph", "in_drive_graph"],
)

print("Saved:", NODES_UNCATEGORIZED_CSV)
nodes_uncategorized.head()


Saved: c:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\notebooks\outputs\nodes_uncategorized.csv


,node_id,base_node_id,x,y,lon,lat,in_walk_graph,in_drive_graph,coord_key
0,245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,True,True,124.2574391|8.183123
1,245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,True,True,124.2451671|8.2122106
2,333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,True,True,124.2439958|8.2243223
3,333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,True,True,124.2442848|8.2248101
4,333797842,333797842,637344.086758,910507.089225,124.246956,8.235139,True,False,124.2469557|8.2351391


## 5. Treat all walk-graph nodes as walk nodes and save `nodes_walk.csv`

In [72]:

nodes_walk = walk_nodes_lookup.copy()[["node_id", "base_node_id", "x", "y", "lon", "lat", "coord_key"]]
node_frame_to_csv(
    nodes_walk,
    NODES_WALK_CSV,
    keep_cols=["node_id", "base_node_id", "x", "y", "lon", "lat"],
)

print("Saved:", NODES_WALK_CSV)
nodes_walk.head()


Saved: c:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\notebooks\outputs\nodes_walk.csv


,node_id,base_node_id,x,y,lon,lat,coord_key
0,245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,124.2574391|8.183123
1,245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,124.2451671|8.2122106
2,333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,124.2439958|8.2243223
3,333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,124.2442848|8.2248101
4,333797842,333797842,637344.086758,910507.089225,124.246956,8.235139,124.2469557|8.2351391


## 6. Treat all drivable nodes as ride nodes and save `nodes_ride.csv`

In [73]:

nodes_ride = drive_nodes_lookup.copy()[["node_id", "base_node_id", "x", "y", "lon", "lat", "coord_key"]]
node_frame_to_csv(
    nodes_ride,
    NODES_RIDE_CSV,
    keep_cols=["node_id", "base_node_id", "x", "y", "lon", "lat"],
)

print("Saved:", NODES_RIDE_CSV)
nodes_ride.head()


Saved: c:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\notebooks\outputs\nodes_ride.csv


,node_id,base_node_id,x,y,lon,lat,coord_key
0,245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,124.2574391|8.183123
1,245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,124.2451671|8.2122106
2,333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,124.2439958|8.2243223
3,333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,124.2442848|8.2248101
4,333801115,333801115,628899.600869,905210.052231,124.170159,8.187466,124.1701587|8.1874658


## 7. Preview the whole-Iligan walk nodes

In [74]:

walk_nodes_preview_gdf = make_nodes_gdf(nodes_walk, crs=G_walk_proj.graph["crs"])
create_interactive_map(
    boundary_gdf=study_area_gdf,
    output_html_path=PREVIEW_WALK_NODES_HTML,
    title="Whole Iligan - Walk Nodes Preview",
    walk_nodes_gdf=walk_nodes_preview_gdf,
)
show_html_preview(PREVIEW_WALK_NODES_HTML)


## 8. Preview the whole-Iligan drivable nodes

In [75]:

ride_nodes_preview_gdf = make_nodes_gdf(nodes_ride, crs=G_drive_proj.graph["crs"])
create_interactive_map(
    boundary_gdf=study_area_gdf,
    output_html_path=PREVIEW_RIDE_NODES_HTML,
    title="Whole Iligan - Drivable Nodes Preview",
    ride_nodes_gdf=ride_nodes_preview_gdf,
)
show_html_preview(PREVIEW_RIDE_NODES_HTML)


## 9. Build the bidirectional start-walk edges

In [76]:

edges_start_walk = graph_edges_to_bidirectional_base(
    G_proj=G_walk_proj,
    prefix="sw",
    edge_type="start_walk",
)
edges_start_walk = assign_edge_ids(edges_start_walk, "sw_edge")
# accessible_nodes not yet computed — saved after Section 19.5

print("Start-walk edges built:", len(edges_start_walk))
edges_start_walk.head()


Start-walk edges built: 11820


,edge_id,u,v,dist,edge_type,geometry
0,sw_edge_000001,sw_245366634,sw_8099897878,24.990711,start_walk,LINESTRING (638516.9708810493 904758.848210290...
1,sw_edge_000002,sw_8099897878,sw_245366634,24.990711,start_walk,LINESTRING (638509.3938900122 904782.532747542...
2,sw_edge_000003,sw_245366634,sw_12599690557,385.609143,start_walk,LINESTRING (638516.9708810493 904758.848210290...
3,sw_edge_000004,sw_12599690557,sw_245366634,385.609143,start_walk,LINESTRING (638479.2545532953 904900.902229016...
4,sw_edge_000005,sw_245366634,sw_12599690568,34.099817,start_walk,LINESTRING (638516.9708810493 904758.848210290...


## 10. Build the bidirectional end-walk edges

In [77]:

edges_end_walk = graph_edges_to_bidirectional_base(
    G_proj=G_walk_proj,
    prefix="ew",
    edge_type="end_walk",
)
edges_end_walk = assign_edge_ids(edges_end_walk, "ew_edge")

print("End-walk edges built:", len(edges_end_walk))
edges_end_walk.head()


End-walk edges built: 11820


,edge_id,u,v,dist,edge_type,geometry
0,ew_edge_000001,ew_245366634,ew_8099897878,24.990711,end_walk,LINESTRING (638516.9708810493 904758.848210290...
1,ew_edge_000002,ew_8099897878,ew_245366634,24.990711,end_walk,LINESTRING (638509.3938900122 904782.532747542...
2,ew_edge_000003,ew_245366634,ew_12599690557,385.609143,end_walk,LINESTRING (638516.9708810493 904758.848210290...
3,ew_edge_000004,ew_12599690557,ew_245366634,385.609143,end_walk,LINESTRING (638479.2545532953 904900.902229016...
4,ew_edge_000005,ew_245366634,ew_12599690568,34.099817,end_walk,LINESTRING (638516.9708810493 904758.848210290...


## 11. Build the bidirectional drivable ride edges

In [78]:

edges_ride = graph_edges_to_bidirectional_base(
    G_proj=G_drive_proj,
    prefix="ride",
    edge_type="ride",
)
edges_ride = assign_edge_ids(edges_ride, "ride_edge")

print("Ride edges built:", len(edges_ride))
edges_ride.head()


Ride edges built: 8834


,edge_id,u,v,dist,edge_type,geometry
0,ride_edge_000001,ride_245366634,ride_1030442070,119.989863,ride,LINESTRING (638528.6552566248 904640.653288030...
1,ride_edge_000002,ride_1030442070,ride_245366634,119.989863,ride,LINESTRING (638516.9708810493 904758.848210290...
2,ride_edge_000003,ride_245366634,ride_8099897878,24.990711,ride,LINESTRING (638516.9708810493 904758.848210290...
3,ride_edge_000004,ride_8099897878,ride_245366634,24.990711,ride,LINESTRING (638509.3938900122 904782.532747542...
4,ride_edge_000005,ride_245366634,ride_12599690567,190.082647,ride,LINESTRING (638516.9708810493 904758.848210290...


## 12. Preview the whole-Iligan drivable network

In [79]:

ride_edges_preview_gdf = make_edges_gdf(edges_ride.copy(), crs=G_drive_proj.graph["crs"])
create_interactive_map(
    boundary_gdf=study_area_gdf,
    output_html_path=PREVIEW_RIDE_NETWORK_HTML,
    title="Whole Iligan - Drivable Network Preview",
    ride_nodes_gdf=ride_nodes_preview_gdf,
    ride_edges_gdf=ride_edges_preview_gdf,
)
show_html_preview(PREVIEW_RIDE_NETWORK_HTML)


## 13. Duplicate the walk nodes into start-walk and end-walk layers

In [80]:

nodes_start_walk, nodes_end_walk = duplicate_walk_nodes_to_layers(nodes_walk)

print("Start-walk nodes:", len(nodes_start_walk))
print("End-walk nodes:", len(nodes_end_walk))
nodes_start_walk.head()


Start-walk nodes: 4689
End-walk nodes: 4689


,node_id,base_node_id,x,y,lon,lat,coord_key
0,sw_245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,124.2574391|8.183123
1,sw_245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,124.2451671|8.2122106
2,sw_333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,124.2439958|8.2243223
3,sw_333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,124.2442848|8.2248101
4,sw_333797842,333797842,637344.086758,910507.089225,124.246956,8.235139,124.2469557|8.2351391


## 14. Rename the drivable nodes into the ride-layer node IDs

In [81]:

nodes_ride_layer = nodes_ride.copy()
nodes_ride_layer["node_id"] = "ride_" + nodes_ride_layer["base_node_id"].astype(str)

print("Ride-layer nodes:", len(nodes_ride_layer))
nodes_ride_layer.head()


Ride-layer nodes: 3477


,node_id,base_node_id,x,y,lon,lat,coord_key
0,ride_245366634,245366634,638516.970881,904758.848210,124.257439,8.183123,124.2574391|8.183123
1,ride_245366694,245366694,637154.936706,907971.088369,124.245167,8.212211,124.2451671|8.2122106
2,ride_333797343,333797343,637021.741374,909309.974211,124.243996,8.224322,124.2439958|8.2243223
3,ride_333797496,333797496,637053.410934,909364.013039,124.244285,8.224810,124.2442848|8.2248101
4,ride_333801115,333801115,628899.600869,905210.052231,124.170159,8.187466,124.1701587|8.1874658


## 15. Build wait edges: `start_walk -> ride`

In [82]:

edges_wait = build_interlayer_edges(
    left_df=nodes_start_walk,
    right_df=nodes_ride_layer,
    left_node_col="node_id",
    right_node_col="node_id",
    edge_type="wait",
    max_snap_dist_m=MAX_INTERLAYER_SNAP_DIST_M,
    anchor_side="right",
)
edges_wait = assign_edge_ids(edges_wait, "wait_edge")

print("Wait edges built:", len(edges_wait))
edges_wait.head()


Wait edges built: 3448


,edge_id,u,v,dist,edge_type,geometry
0,wait_edge_000001,sw_245366634,ride_245366634,0.0,wait,LINESTRING (638516.9708810493 904758.848210290...
1,wait_edge_000002,sw_245366694,ride_245366694,0.0,wait,LINESTRING (637154.9367063687 907971.088368548...
2,wait_edge_000003,sw_333797343,ride_333797343,0.0,wait,LINESTRING (637021.7413736106 909309.974211277...
3,wait_edge_000004,sw_333797496,ride_333797496,0.0,wait,LINESTRING (637053.4109339686 909364.013038886...
4,wait_edge_000005,sw_333801115,ride_333801115,0.0,wait,LINESTRING (628899.6008685874 905210.052231287...


## 16. Build alight edges: `ride -> end_walk`

In [83]:

edges_alight = build_interlayer_edges(
    left_df=nodes_ride_layer,
    right_df=nodes_end_walk,
    left_node_col="node_id",
    right_node_col="node_id",
    edge_type="alight",
    max_snap_dist_m=MAX_INTERLAYER_SNAP_DIST_M,
    anchor_side="left",
)
edges_alight = assign_edge_ids(edges_alight, "alight_edge")

print("Alight edges built:", len(edges_alight))
edges_alight.head()


Alight edges built: 3448


,edge_id,u,v,dist,edge_type,geometry
0,alight_edge_000001,ride_245366634,ew_245366634,0.0,alight,LINESTRING (638516.9708810493 904758.848210290...
1,alight_edge_000002,ride_245366694,ew_245366694,0.0,alight,LINESTRING (637154.9367063687 907971.088368548...
2,alight_edge_000003,ride_333797343,ew_333797343,0.0,alight,LINESTRING (637021.7413736106 909309.974211277...
3,alight_edge_000004,ride_333797496,ew_333797496,0.0,alight,LINESTRING (637053.4109339686 909364.013038886...
4,alight_edge_000005,ride_333801115,ew_333801115,0.0,alight,LINESTRING (628899.6008685874 905210.052231287...


## 17. Build direct edges: `start_walk -> end_walk`

In [84]:

edges_direct = build_direct_edges(nodes_start_walk, nodes_end_walk)
edges_direct = assign_edge_ids(edges_direct, "direct_edge")

print("Direct edges built:", len(edges_direct))
edges_direct.head()


Direct edges built: 4689


,edge_id,u,v,dist,edge_type,geometry
0,direct_edge_000001,sw_245366634,ew_245366634,0.0,direct,LINESTRING (638516.9708810493 904758.848210290...
1,direct_edge_000002,sw_245366694,ew_245366694,0.0,direct,LINESTRING (637154.9367063687 907971.088368548...
2,direct_edge_000003,sw_333797343,ew_333797343,0.0,direct,LINESTRING (637021.7413736106 909309.974211277...
3,direct_edge_000004,sw_333797496,ew_333797496,0.0,direct,LINESTRING (637053.4109339686 909364.013038886...
4,direct_edge_000005,sw_333797842,ew_333797842,0.0,direct,LINESTRING (637344.0867584045 910507.089225195...


## 18. Build transfer edges: `end_walk -> ride`

In [85]:

edges_transfer = build_interlayer_edges(
    left_df=nodes_end_walk,
    right_df=nodes_ride_layer,
    left_node_col="node_id",
    right_node_col="node_id",
    edge_type="transfer",
    max_snap_dist_m=MAX_INTERLAYER_SNAP_DIST_M,
    anchor_side="right",
)
edges_transfer = assign_edge_ids(edges_transfer, "transfer_edge")

print("Transfer edges built:", len(edges_transfer))
edges_transfer.head()


Transfer edges built: 3448


,edge_id,u,v,dist,edge_type,geometry
0,transfer_edge_000001,ew_245366634,ride_245366634,0.0,transfer,LINESTRING (638516.9708810493 904758.848210290...
1,transfer_edge_000002,ew_245366694,ride_245366694,0.0,transfer,LINESTRING (637154.9367063687 907971.088368548...
2,transfer_edge_000003,ew_333797343,ride_333797343,0.0,transfer,LINESTRING (637021.7413736106 909309.974211277...
3,transfer_edge_000004,ew_333797496,ride_333797496,0.0,transfer,LINESTRING (637053.4109339686 909364.013038886...
4,transfer_edge_000005,ew_333801115,ride_333801115,0.0,transfer,LINESTRING (628899.6008685874 905210.052231287...


## 19. Stitch the graph together

In [86]:

travel_graph_nodes = pd.concat(
    [
        nodes_start_walk.assign(layer="start_walk"),
        nodes_ride_layer.assign(layer="ride"),
        nodes_end_walk.assign(layer="end_walk"),
    ],
    ignore_index=True,
    sort=False,
)[["node_id", "base_node_id", "x", "y", "lon", "lat", "layer"]].drop_duplicates("node_id").reset_index(drop=True)

# Concatenate all seven edge types into the master DataFrame.
# accessible_nodes will be attached in Section 19.5 before any CSV is written.
travel_graph_edges = pd.concat(
    [
        edges_start_walk,
        edges_end_walk,
        edges_ride,
        edges_wait,
        edges_alight,
        edges_direct,
        edges_transfer,
    ],
    ignore_index=True,
    sort=False,
)

print("Total layered nodes:", len(travel_graph_nodes))
print("Total layered edges:", len(travel_graph_edges))
travel_graph_edges.head()


Total layered nodes: 12855
Total layered edges: 47507


,edge_id,u,v,dist,edge_type,geometry
0,sw_edge_000001,sw_245366634,sw_8099897878,24.990711,start_walk,LINESTRING (638516.9708810493 904758.848210290...
1,sw_edge_000002,sw_8099897878,sw_245366634,24.990711,start_walk,LINESTRING (638509.3938900122 904782.532747542...
2,sw_edge_000003,sw_245366634,sw_12599690557,385.609143,start_walk,LINESTRING (638516.9708810493 904758.848210290...
3,sw_edge_000004,sw_12599690557,sw_245366634,385.609143,start_walk,LINESTRING (638479.2545532953 904900.902229016...
4,sw_edge_000005,sw_245366634,sw_12599690568,34.099817,start_walk,LINESTRING (638516.9708810493 904758.848210290...


## 19.5. Compute `accessible_nodes` and save all CSVs

**NEW** — must run after Section 19 (all seven edge types must be stitched first).

`accessible_nodes` for edge `(u→v)` = every `edge_id` where the next edge's `u == this edge's v`.
Stored as a semicolon-delimited string. Empty string = terminal node (valid trip endpoint).

The lookup is built once from the full graph, then applied to every per-type DataFrame
and to the master stitched DataFrame before any CSV is written.

In [87]:

# Step 1: build the v → [edge_ids] lookup from the full stitched graph
v_to_outgoing = compute_v_to_outgoing(travel_graph_edges)

# Step 2: attach accessible_nodes to the master stitched DataFrame
travel_graph_edges = attach_accessible_nodes(travel_graph_edges, v_to_outgoing)

# Step 3: also attach to each per-type DataFrame and propagate back
edges_start_walk = attach_accessible_nodes(edges_start_walk, v_to_outgoing)
edges_end_walk   = attach_accessible_nodes(edges_end_walk,   v_to_outgoing)
edges_ride       = attach_accessible_nodes(edges_ride,       v_to_outgoing)
edges_wait       = attach_accessible_nodes(edges_wait,       v_to_outgoing)
edges_alight     = attach_accessible_nodes(edges_alight,     v_to_outgoing)
edges_direct     = attach_accessible_nodes(edges_direct,     v_to_outgoing)
edges_transfer   = attach_accessible_nodes(edges_transfer,   v_to_outgoing)

# Step 4: save all per-type CSVs (edge_frame_to_csv now includes accessible_nodes)
_EDGE_LAYERS = [
    (edges_start_walk, EDGES_START_WALK_CSV, "edges_start_walk"),
    (edges_end_walk,   EDGES_END_WALK_CSV,   "edges_end_walk"),
    (edges_ride,       EDGES_RIDE_CSV,       "edges_ride"),
    (edges_wait,       EDGES_WAIT_CSV,       "edges_wait"),
    (edges_alight,     EDGES_ALIGHT_CSV,     "edges_alight"),
    (edges_direct,     EDGES_DIRECT_CSV,     "edges_direct"),
    (edges_transfer,   EDGES_TRANSFER_CSV,   "edges_transfer"),
]
for _df, _csv_path, _label in _EDGE_LAYERS:
    edge_frame_to_csv(_df, _csv_path)
    print(f"Saved {_label:<20} → {_csv_path.name}  ({len(_df):,} rows)")

# Step 5: save the master stitched CSV and node CSV
travel_graph_nodes.to_csv(TRAVEL_GRAPH_NODES_CSV, index=False)
edge_frame_to_csv(travel_graph_edges, TRAVEL_GRAPH_EDGES_CSV)
print(f"\nSaved travel_graph_nodes    → {TRAVEL_GRAPH_NODES_CSV.name}  ({len(travel_graph_nodes):,} rows)")
print(f"Saved travel_graph_edges    → {TRAVEL_GRAPH_EDGES_CSV.name}  ({len(travel_graph_edges):,} rows)")

# Step 6: sanity checks
_n_total          = len(travel_graph_edges)
_n_with_acc       = (travel_graph_edges["accessible_nodes"] != "").sum()
_n_empty          = _n_total - _n_with_acc
print(f"\n--- accessible_nodes sanity ---")
print(f"  Edges with ≥1 accessible edge  : {_n_with_acc:,} / {_n_total:,}")
print(f"  Terminal edges (empty list)    : {_n_empty:,}")

_empty_by_type = (
    travel_graph_edges[travel_graph_edges["accessible_nodes"] == ""]
    .groupby("edge_type")
    .size()
    .rename("n_terminal")
)
print("\n  Terminal edges by type:")
print(_empty_by_type.to_string())

# Spot-check: a wait edge's v is a ride node → accessible edges must be ride/alight only
_sample_wait = travel_graph_edges[travel_graph_edges["edge_type"] == "wait"].iloc[0]
_acc_ids      = [e for e in _sample_wait["accessible_nodes"].split(";") if e]
_acc_types    = (
    travel_graph_edges[travel_graph_edges["edge_id"].isin(_acc_ids)]["edge_type"]
    .unique().tolist()
)
print(f"\n  Spot-check — first wait edge ({_sample_wait['edge_id']}):")
print(f"    v = {_sample_wait['v']}")
print(f"    accessible edge types: {_acc_types}")
assert all(t in {"ride", "alight"} for t in _acc_types), (
    "Wait edge should only lead to ride/alight edges — check interlayer connections."
)
print("  ✓ Wait edge leads only to ride/alight edges.")


Saved edges_start_walk     → edges_start_walk.csv  (11,820 rows)
Saved edges_end_walk       → edges_end_walk.csv  (11,820 rows)
Saved edges_ride           → edges_ride.csv  (8,834 rows)
Saved edges_wait           → edges_wait.csv  (3,448 rows)
Saved edges_alight         → edges_alight.csv  (3,448 rows)
Saved edges_direct         → edges_direct.csv  (4,689 rows)
Saved edges_transfer       → edges_transfer.csv  (3,448 rows)

Saved travel_graph_nodes    → travel_graph_nodes.csv  (12,855 rows)
Saved travel_graph_edges    → iligan_travel_graph.csv  (47,507 rows)

--- accessible_nodes sanity ---
  Edges with ≥1 accessible edge  : 47,503 / 47,507
  Terminal edges (empty list)    : 4

  Terminal edges by type:
edge_type
direct    4

  Spot-check — first wait edge (wait_edge_000001):
    v = ride_245366634
    accessible edge types: ['ride', 'alight']
  ✓ Wait edge leads only to ride/alight edges.


## 20. Build the in-memory directed graph object

In [88]:

travel_graph_nx = nx.DiGraph()
add_nodes_to_digraph(travel_graph_nx, travel_graph_nodes)
add_edges_to_digraph(travel_graph_nx, travel_graph_edges)

print("Directed graph nodes:", travel_graph_nx.number_of_nodes())
print("Directed graph edges:", travel_graph_nx.number_of_edges())


Directed graph nodes: 12855
Directed graph edges: 47463


## 21. Interactive stitched preview for verification

In [ ]:
PREVIEW_STITCHED_HTML = PREVIEW_DIR / "preview_travel_graph_stitched.html"


## 22. Validation summary

In [90]:

validation_summary = pd.DataFrame(
    {
        "component": [
            "nodes_uncategorized",
            "nodes_walk",
            "nodes_ride",
            "edges_start_walk",
            "edges_end_walk",
            "edges_ride",
            "edges_wait",
            "edges_alight",
            "edges_direct",
            "edges_transfer",
            "travel_graph_nodes",
            "travel_graph_edges",
        ],
        "count": [
            len(nodes_uncategorized),
            len(nodes_walk),
            len(nodes_ride),
            len(edges_start_walk),
            len(edges_end_walk),
            len(edges_ride),
            len(edges_wait),
            len(edges_alight),
            len(edges_direct),
            len(edges_transfer),
            len(travel_graph_nodes),
            len(travel_graph_edges),
        ],
    }
)
validation_summary


,component,count
0,nodes_uncategorized,4758
1,nodes_walk,4689
2,nodes_ride,3477
3,edges_start_walk,11820
4,edges_end_walk,11820
5,edges_ride,8834
6,edges_wait,3448
7,edges_alight,3448
8,edges_direct,4689
9,edges_transfer,3448


## 23. `TravelGraphManager`

**NEW / UPDATED** — initialise the manager with the stitched CSV, your survey-derived
β coefficients, and optionally a set of `JeepneyRoute` objects.

### Route filtering
When `routes` are supplied the graph is pruned so only the specified jeepney roads
are reachable:

| Edge type | Kept when routes provided |
|-----------|--------------------------|
| `start_walk` / `end_walk` / `direct` | always (passenger walks freely) |
| `ride` | only if `(u, v)` is an edge pair of at least one route |
| `wait` | only if `v` (ride node) appears in at least one route |
| `alight` | only if `u` (ride node) appears in at least one route |
| `transfer` | only if `v` (ride node) appears in at least one route |

### Weight parameters
| Parameter | Symbol | Applied to |
|-----------|--------|------------|
| `walk_wt` | β_walk | `start_walk` / `end_walk`: `dist_m × β_walk` |
| `ride_wt` | β_ride | `ride`: `dist_m × β_ride` |
| `wait_wt` | β_wait | `wait`: flat boarding penalty |
| `transfer_wt` | β_transfer | `transfer`: flat vehicle-change penalty |
| — | 0 | `alight` / `direct`: zero by design |

In [91]:

import random
import warnings
import json as _json
from pathlib import Path as _Path


# ── Edge-type constants ──────────────────────────────────────────────────────

_WALK_TYPES    = {"start_walk", "end_walk"}
_RIDE_TYPE     = "ride"
_WAIT_TYPE     = "wait"
_ALIGHT_TYPE   = "alight"
_DIRECT_TYPE   = "direct"
_TRANSFER_TYPE = "transfer"
_ALL_KNOWN_TYPES = _WALK_TYPES | {_RIDE_TYPE, _WAIT_TYPE, _ALIGHT_TYPE,
                                   _DIRECT_TYPE, _TRANSFER_TYPE}
_REQUIRED_COLUMNS = {"edge_id", "u", "v", "dist", "edge_type"}

# Colour palette for route visualisation (Leaflet hex strings)
_ROUTE_COLOURS = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00",
    "#a65628", "#f781bf", "#999999", "#66c2a5", "#fc8d62",
]


# ── TravelGraphManager ───────────────────────────────────────────────────────

class TravelGraphManager:
    """
    Loads the stitched travel graph and exposes routing + visualisation.

    Parameters
    ----------
    edges_csv : str or Path
        Path to the stitched edges CSV (iligan_travel_graph.csv).
    nodes_csv : str or Path, optional
        Path to the stitched nodes CSV (travel_graph_nodes.csv).
        Required for visualise_path(); contains lat/lon per node.
    routes : list[JeepneyRoute], optional
        If provided the ride layer is restricted to these routes only.
        If None the full ride graph is used (useful for inspection).
    walk_wt : float   β_walk  — multiplied by dist_m for walk edges.
    ride_wt : float   β_ride  — multiplied by dist_m for ride edges.
    wait_wt : float   β_wait  — flat boarding disutility.
    transfer_wt : float  β_transfer — flat vehicle-change disutility.
    """

    def __init__(
        self,
        edges_csv,
        nodes_csv=None,
        routes=None,
        *,
        walk_wt: float,
        ride_wt: float,
        wait_wt: float,
        transfer_wt: float,
    ) -> None:
        self._walk_wt     = float(walk_wt)
        self._ride_wt     = float(ride_wt)
        self._wait_wt     = float(wait_wt)
        self._transfer_wt = float(transfer_wt)
        self._routes      = list(routes) if routes is not None else None

        self._edges_df  = self._load_edges(_Path(edges_csv))
        self._nodes_df  = self._load_nodes(_Path(nodes_csv)) if nodes_csv else None

        # Apply route filtering if routes were supplied
        self._active_edges = (
            self._filter_by_routes(self._edges_df, self._routes)
            if self._routes is not None
            else self._edges_df
        )

        self._graph      = self._build_weighted_graph(self._active_edges)
        self._edge_by_id = {
            row.edge_id: row
            for row in self._active_edges.itertuples(index=False)
        }
        self._accessible  = self._build_accessible_index()
        self._node_coords = self._build_node_coords()

        self._sanity_check()

    # ── Private: loading ─────────────────────────────────────────────────────

    @staticmethod
    def _load_edges(path: _Path) -> pd.DataFrame:
        if not path.exists():
            raise FileNotFoundError(f"Edges CSV not found: {path}")
        df = pd.read_csv(
            path,
            dtype={"edge_id": str, "u": str, "v": str, "edge_type": str},
        )
        missing = _REQUIRED_COLUMNS - set(df.columns)
        if missing:
            raise ValueError(f"Edges CSV missing required columns: {missing}")
        return df

    @staticmethod
    def _load_nodes(path: _Path) -> pd.DataFrame:
        if not path.exists():
            raise FileNotFoundError(f"Nodes CSV not found: {path}")
        return pd.read_csv(
            path,
            dtype={"node_id": str},
        )

    # ── Private: route filtering ─────────────────────────────────────────────

    @staticmethod
    def _filter_by_routes(df: pd.DataFrame, routes: list) -> pd.DataFrame:
        """
        Keep only the edges a passenger can actually use given the supplied routes.

        Rules per edge type
        -------------------
        start_walk / end_walk / direct : always kept (passenger walks freely).
        ride       : kept only if (u, v) is an edge pair of at least one route.
        wait       : kept only if v (ride node) is in at least one route.
        alight     : kept only if u (ride node) is in at least one route.
        transfer   : kept only if v (ride node) is in at least one route.
        """
        valid_pairs = set()
        valid_nodes = set()
        for route in routes:
            valid_pairs |= route.edge_pairs
            valid_nodes |= route.node_set

        def _keep(row) -> bool:
            et = row.edge_type
            if et in _WALK_TYPES or et == _DIRECT_TYPE:
                return True
            if et == _RIDE_TYPE:
                return (row.u, row.v) in valid_pairs
            if et == _WAIT_TYPE:
                return row.v in valid_nodes      # sw → ride: is the stop on a route?
            if et == _ALIGHT_TYPE:
                return row.u in valid_nodes      # ride → ew: is the stop on a route?
            if et == _TRANSFER_TYPE:
                return row.v in valid_nodes      # ew → ride: is the stop on a route?
            return True

        mask = df.apply(_keep, axis=1)
        return df[mask].reset_index(drop=True)

    # ── Private: graph + index construction ─────────────────────────────────

    def _edge_weight(self, edge_type: str, dist: float) -> float:
        if edge_type in _WALK_TYPES:            return dist * self._walk_wt
        if edge_type == _RIDE_TYPE:             return dist * self._ride_wt
        if edge_type == _WAIT_TYPE:             return self._wait_wt
        if edge_type == _TRANSFER_TYPE:         return self._transfer_wt
        if edge_type in (_ALIGHT_TYPE, _DIRECT_TYPE): return 0.0
        warnings.warn(f"Unknown edge_type {edge_type!r}; using raw dist.", stacklevel=2)
        return dist

    def _build_weighted_graph(self, df: pd.DataFrame) -> nx.DiGraph:
        G = nx.DiGraph()
        for row in df.itertuples(index=False):
            G.add_edge(
                row.u, row.v,
                edge_id=row.edge_id,
                weight=self._edge_weight(row.edge_type, float(row.dist)),
                edge_type=row.edge_type,
                dist=float(row.dist),
            )
        return G

    def _build_accessible_index(self) -> dict:
        """
        Build edge_id → [reachable edge_ids] from the ACTIVE (possibly filtered) graph.
        Reads the accessible_nodes CSV column if present; otherwise derives from graph.
        """
        if "accessible_nodes" in self._active_edges.columns:
            active_ids = set(self._active_edges["edge_id"])
            idx = {}
            for row in self._active_edges.itertuples(index=False):
                raw = getattr(row, "accessible_nodes", "") or ""
                # Only include neighbours that are still in the active (filtered) graph
                idx[row.edge_id] = [
                    e for e in str(raw).split(";")
                    if e and e in active_ids
                ]
            return idx
        warnings.warn(
            "accessible_nodes column not found — computing from filtered graph.",
            stacklevel=2,
        )
        v_to_out: dict = {}
        for u, v, data in self._graph.edges(data=True):
            v_to_out.setdefault(u, []).append(data["edge_id"])
        return {
            row.edge_id: v_to_out.get(row.v, [])
            for row in self._active_edges.itertuples(index=False)
        }

    def _build_node_coords(self) -> dict:
        """Build node_id → (lat, lon) from the nodes DataFrame (if loaded)."""
        if self._nodes_df is None:
            return {}
        required = {"node_id", "lat", "lon"}
        if not required.issubset(self._nodes_df.columns):
            warnings.warn(
                f"nodes CSV missing one of {required}; visualisation disabled.",
                stacklevel=2,
            )
            return {}
        return {
            row.node_id: (float(row.lat), float(row.lon))
            for row in self._nodes_df.itertuples(index=False)
        }

    def _sanity_check(self) -> None:
        assert self._graph.number_of_nodes() > 0, "Graph has no nodes."
        assert self._graph.number_of_edges() > 0, "Graph has no edges."
        actual_types = set(self._active_edges["edge_type"].unique())
        unknown = actual_types - _ALL_KNOWN_TYPES
        if unknown:
            warnings.warn(f"Unexpected edge_type(s): {unknown}")
        route_str = (
            f"{len(self._routes)} routes"
            if self._routes is not None else "full ride graph"
        )
        print(
            f"[TravelGraphManager] {route_str} | "
            f"{self._graph.number_of_nodes():,} nodes | "
            f"{self._graph.number_of_edges():,} edges"
        )
        print(f"  Edge types : {sorted(actual_types)}")
        print(
            f"  Weights    — walk: {self._walk_wt}, ride: {self._ride_wt}, "
            f"wait: {self._wait_wt}, transfer: {self._transfer_wt}"
        )

    # ── Getters ──────────────────────────────────────────────────────────────

    @property
    def edges(self) -> pd.DataFrame:
        """Active (possibly route-filtered) edges DataFrame (copy)."""
        return self._active_edges.copy()

    @property
    def graph(self) -> nx.DiGraph:
        """Underlying weighted NetworkX DiGraph (active edges only)."""
        return self._graph

    @property
    def routes(self):
        """List of JeepneyRoute objects, or None if using full ride graph."""
        return list(self._routes) if self._routes is not None else None

    @property
    def walk_wt(self) -> float:     return self._walk_wt
    @property
    def ride_wt(self) -> float:     return self._ride_wt
    @property
    def wait_wt(self) -> float:     return self._wait_wt
    @property
    def transfer_wt(self) -> float: return self._transfer_wt

    def get_edge(self, edge_id: str):
        return self._edge_by_id.get(edge_id)

    def get_edges_from_node(self, node_id: str) -> pd.DataFrame:
        return self._active_edges[self._active_edges["u"] == node_id].copy()

    def get_edges_to_node(self, node_id: str) -> pd.DataFrame:
        return self._active_edges[self._active_edges["v"] == node_id].copy()

    def get_accessible_edges(self, edge_id: str) -> list:
        """
        Edge_ids reachable after traversing edge_id. Reflects the active
        (route-filtered) graph — O(1) lookup.
        """
        return list(self._accessible.get(edge_id, []))

    # ── Method 1: generate_random_ride_loop ─────────────────────────────────

    def generate_random_ride_loop(
        self,
        min_length: int,
        max_length: int,
        seed=None,
    ) -> list:
        """
        Generate a random cycle within the *active* ride layer.

        Useful for creating synthetic JeepneyRoute objects for testing:
            nodes = mgr.generate_random_ride_loop(5, 20)
            route = JeepneyRoute("test", nodes[:-1])  # drop repeated last node

        Returns
        -------
        list[str]
            Ride node IDs forming a closed cycle (first == last element).
        """
        if min_length < 2:
            raise ValueError("min_length must be >= 2.")
        if min_length > max_length:
            raise ValueError("min_length must be <= max_length.")

        rng = random.Random(seed)
        ride_df = self._active_edges[self._active_edges["edge_type"] == _RIDE_TYPE]
        if ride_df.empty:
            raise ValueError(
                "No ride edges in active graph. "
                "If routes were provided, ensure they contain valid ride edge pairs."
            )

        ride_G = nx.DiGraph()
        for row in ride_df.itertuples(index=False):
            ride_G.add_edge(row.u, row.v)

        ride_nodes = list(ride_G.nodes())
        if not ride_nodes:
            raise ValueError("No ride nodes found in the active ride sub-graph.")

        for _ in range(300):
            start   = rng.choice(ride_nodes)
            path    = [start]
            current = start
            visited = {start}
            target  = rng.randint(min_length, max_length)

            while len(path) < target:
                unvisited = [n for n in ride_G.successors(current) if n not in visited]
                nbrs = unvisited or list(ride_G.successors(current))
                if not nbrs:
                    break
                current = rng.choice(nbrs)
                path.append(current)
                visited.add(current)

            if len(path) < min_length:
                continue

            if ride_G.has_edge(current, start):
                return path + [start]

            try:
                ret = nx.shortest_path(ride_G, current, start)
                combined = path + ret[1:]
                if len(combined) - 1 >= min_length:
                    return combined
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                pass

        raise ValueError(
            f"Could not generate a ride loop of length [{min_length}, {max_length}] "
            "after 300 attempts."
        )

    # ── Method 2: calculate_shortest_path ───────────────────────────────────

    def calculate_shortest_path(self, u: str, v: str) -> list:
        """
        Minimum-disutility path from node u to node v using Dijkstra.

        Only active (route-filtered) edges are considered.  A passenger
        restricted to routes R1 and R2 cannot ride edges belonging only to R3.

        Returns
        -------
        list[str]  Ordered edge_ids traversed. Empty list if u == v.

        Raises
        ------
        nx.NodeNotFound    If u or v is absent from the active graph.
        nx.NetworkXNoPath  If no path exists (e.g. destination unreachable
                           with the given routes).
        """
        if u not in self._graph:
            raise nx.NodeNotFound(f"Origin not in active graph: {u!r}")
        if v not in self._graph:
            raise nx.NodeNotFound(f"Destination not in active graph: {v!r}")
        if u == v:
            return []
        node_path = nx.dijkstra_path(self._graph, u, v, weight="weight")
        return [self._graph[a][b]["edge_id"] for a, b in zip(node_path[:-1], node_path[1:])]

    # ── Method 3: visualize_path ─────────────────────────────────────────────

    def visualize_path(
        self,
        path_edges: list,
        output_html,
        title: str = "Jeepney Journey",
    ) -> _Path:
        """
        Generate a self-contained Leaflet HTML map of a journey.

        Shows:
        - All active jeepney routes as coloured polylines (toggleable).
        - The journey path highlighted by segment type (walk / ride / transfer).
        - Origin (green) and destination (red) markers with popups.
        - A legend and layer control.

        Parameters
        ----------
        path_edges : list[str]
            Output of calculate_shortest_path().
        output_html : str or Path
            Where to write the HTML file.
        title : str
            Map title shown in the top-left panel.

        Returns
        -------
        Path  Absolute path to the generated HTML file.

        Raises
        ------
        RuntimeError  If nodes_csv was not provided at initialisation.
        """
        if not self._node_coords:
            raise RuntimeError(
                "Node coordinates are unavailable. "
                "Pass nodes_csv= when creating TravelGraphManager."
            )
        if not path_edges:
            raise ValueError("path_edges is empty — nothing to visualise.")

        out_path = _Path(output_html).resolve()

        # ── Collect journey segments ──────────────────────────────────────
        SEGMENT_COLOURS = {
            "start_walk": "#1f77b4",   # blue
            "end_walk":   "#1f77b4",
            "direct":     "#7f7f7f",   # grey
            "ride":       "#d62728",   # red  (overridden by route colour below)
            "wait":       "#2ca02c",   # green
            "alight":     "#9467bd",   # purple
            "transfer":   "#ff7f0e",   # orange
        }

        # Map ride edge_id → route colour
        ride_edge_to_colour = {}
        if self._routes:
            for idx, route in enumerate(self._routes):
                colour = _ROUTE_COLOURS[idx % len(_ROUTE_COLOURS)]
                for u_node, v_node in route.edge_pairs:
                    # Find the matching edge_id
                    matches = self._active_edges[
                        (self._active_edges["u"] == u_node) &
                        (self._active_edges["v"] == v_node) &
                        (self._active_edges["edge_type"] == _RIDE_TYPE)
                    ]["edge_id"]
                    for eid in matches:
                        ride_edge_to_colour[eid] = colour

        segments = []
        for eid in path_edges:
            row = self.get_edge(eid)
            if row is None:
                continue
            u_coord = self._node_coords.get(row.u)
            v_coord = self._node_coords.get(row.v)
            if u_coord is None or v_coord is None:
                continue
            et = row.edge_type
            colour = (
                ride_edge_to_colour.get(eid, SEGMENT_COLOURS["ride"])
                if et == _RIDE_TYPE
                else SEGMENT_COLOURS.get(et, "#333333")
            )
            segments.append({
                "coords": [list(u_coord), list(v_coord)],
                "edge_id": eid,
                "edge_type": et,
                "dist": round(float(row.dist), 1),
                "colour": colour,
            })

        if not segments:
            raise RuntimeError(
                "No segments could be drawn — node coordinates may be missing for "
                "the nodes in this path."
            )

        origin = segments[0]["coords"][0]
        dest   = segments[-1]["coords"][1]

        # ── Build route polyline data ─────────────────────────────────────
        route_layers = []
        if self._routes:
            for idx, route in enumerate(self._routes):
                colour = _ROUTE_COLOURS[idx % len(_ROUTE_COLOURS)]
                coords = []
                for nid in route.nodes:
                    c = self._node_coords.get(nid)
                    if c:
                        coords.append(list(c))
                # close the loop visually
                if coords:
                    coords.append(coords[0])
                route_layers.append({
                    "id":     route.route_id,
                    "colour": colour,
                    "coords": coords,
                })

        # ── Summary stats ─────────────────────────────────────────────────
        type_counts = {}
        for s in segments:
            type_counts[s["edge_type"]] = type_counts.get(s["edge_type"], 0) + 1
        walk_dist = sum(
            s["dist"] for s in segments
            if s["edge_type"] in _WALK_TYPES or s["edge_type"] == _DIRECT_TYPE
        )
        ride_dist = sum(s["dist"] for s in segments if s["edge_type"] == _RIDE_TYPE)
        n_transfers = type_counts.get("transfer", 0)

        summary_html = (
            f"<b>Walk</b> {walk_dist:.0f} m &nbsp;|&nbsp; "
            f"<b>Ride</b> {ride_dist:.0f} m &nbsp;|&nbsp; "
            f"<b>Transfers</b> {n_transfers}"
        )

        # ── Build legend HTML ─────────────────────────────────────────────
        legend_items = ""
        seen = set()
        for s in segments:
            et = s["edge_type"]
            c  = s["colour"]
            if et not in seen:
                seen.add(et)
                legend_items += (
                    f'<div style="display:flex;align-items:center;gap:6px;margin:2px 0">' +
                    f'<span style="display:inline-block;width:24px;height:4px;' +
                    f'background:{c};border-radius:2px"></span>' +
                    f'<span style="font-size:12px">{et.replace("_"," ")}</span></div>'
                )
        if self._routes:
            for rl in route_layers:
                legend_items += (
                    f'<div style="display:flex;align-items:center;gap:6px;margin:2px 0">' +
                    f'<span style="display:inline-block;width:24px;height:4px;opacity:0.5;' +
                    f'background:{rl["colour"]};border-radius:2px"></span>' +
                    f'<span style="font-size:12px">Route {rl["id"]}</span></div>'
                )

        # ── Embed data + write HTML ───────────────────────────────────────
        segments_json    = _json.dumps(segments)
        routes_json      = _json.dumps(route_layers)
        origin_json      = _json.dumps(origin)
        dest_json        = _json.dumps(dest)

        html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<title>{title}</title>
<meta name="viewport" content="width=device-width,initial-scale=1"/>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
  html,body {{margin:0;padding:0;height:100%;font-family:sans-serif}}
  #map {{height:100vh;width:100%}}
  #panel {{
    position:fixed;top:12px;left:50%;transform:translateX(-50%);
    z-index:9999;background:rgba(255,255,255,0.96);
    border-radius:8px;box-shadow:0 2px 10px rgba(0,0,0,0.25);
    padding:10px 16px;max-width:500px;text-align:center;
  }}
  #panel h3 {{margin:0 0 4px;font-size:15px}}
  #panel p  {{margin:0;font-size:12px;color:#555}}
  #legend {{
    position:fixed;bottom:28px;right:12px;z-index:9999;
    background:rgba(255,255,255,0.95);border-radius:8px;
    box-shadow:0 2px 8px rgba(0,0,0,0.2);padding:10px 14px;min-width:140px;
  }}
  #legend h4 {{margin:0 0 6px;font-size:13px}}
</style>
</head>
<body>
<div id="panel">
  <h3>{title}</h3>
  <p>{summary_html}</p>
</div>
<div id="legend">
  <h4>Legend</h4>
  {legend_items}
</div>
<div id="map"></div>
<script>
const map = L.map("map");
L.tileLayer("https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png",{{
  attribution: "&copy; OpenStreetMap contributors",maxZoom:19
}}).addTo(map);

const routes   = {routes_json};
const segments = {segments_json};
const origin   = {origin_json};
const dest     = {dest_json};

// ── Draw jeepney routes (background, semi-transparent) ──────────────────
const routeGroup = L.layerGroup().addTo(map);
routes.forEach(r => {{
  if (r.coords.length < 2) return;
  L.polyline(r.coords, {{
    color: r.colour, weight: 4, opacity: 0.35, dashArray: "8 5"
  }}).bindPopup(`<b>Route ${{r.id}}</b>`).addTo(routeGroup);
}});

// ── Draw journey segments ────────────────────────────────────────────────
const journeyGroup = L.layerGroup().addTo(map);
segments.forEach(s => {{
  const weight = (s.edge_type === "ride") ? 6 : 4;
  const popup  = `<b>${{s.edge_type.replace(/_/g," ")}}</b><br/>` +
                 `ID: ${{s.edge_id}}<br/>dist: ${{s.dist}} m`;
  L.polyline(s.coords, {{
    color: s.colour, weight: weight, opacity: 0.9
  }}).bindPopup(popup).addTo(journeyGroup);
}});

// ── Origin / destination markers ────────────────────────────────────────
const greenIcon = L.divIcon({{className:"",html:
  '<div style="width:14px;height:14px;border-radius:50%;background:#2ca02c;' +
  'border:2px solid #fff;box-shadow:0 0 4px rgba(0,0,0,0.5)"></div>'
}});
const redIcon = L.divIcon({{className:"",html:
  '<div style="width:14px;height:14px;border-radius:50%;background:#d62728;' +
  'border:2px solid #fff;box-shadow:0 0 4px rgba(0,0,0,0.5)"></div>'
}});
L.marker(origin, {{icon:greenIcon}}).bindPopup("<b>Origin</b>").addTo(map);
L.marker(dest,   {{icon:redIcon  }}).bindPopup("<b>Destination</b>").addTo(map);

// ── Layer control ────────────────────────────────────────────────────────
L.control.layers(null, {{
  "Jeepney routes": routeGroup,
  "Journey path":   journeyGroup,
}}, {{collapsed:false, position:"topright"}}).addTo(map);

// ── Fit bounds to journey ────────────────────────────────────────────────
const allCoords = segments.flatMap(s => s.coords);
if (allCoords.length) map.fitBounds(allCoords, {{padding:[40,40]}});
</script>
</body>
</html>"""

        out_path.write_text(html, encoding="utf-8")
        print(f"Saved journey map → {out_path}")
        return out_path

    def __repr__(self) -> str:
        route_str = (
            f"{len(self._routes)} routes"
            if self._routes is not None else "full ride graph"
        )
        return (
            f"TravelGraphManager("
            f"{route_str}, "
            f"nodes={self._graph.number_of_nodes():,}, "
            f"edges={self._graph.number_of_edges():,})"
        )


## 24. Instantiate `TravelGraphManager` with jeepney routes

The cell below generates two synthetic routes using `generate_random_ride_loop`,
wraps them in `JeepneyRoute` objects, then restarts the manager restricted to those routes.

**Replace the synthetic routes with real ones** once you have surveyed route data.

In [97]:

# ── Step 1: spin up a temporary full-ride manager to sample route loops ──────
_mgr_full = TravelGraphManager(
    edges_csv   = TRAVEL_GRAPH_EDGES_CSV,
    walk_wt     = 0.0142,
    ride_wt     = 0.0071,
    wait_wt     = 8.5,
    transfer_wt = 14.2,
)

# ── Step 2: generate two synthetic jeepney routes ─────────────────────────────
# Each call returns a closed list of ride nodes [first, ..., first].
# Drop the last element (duplicate of first) before passing to JeepneyRoute.
_loop1 = _mgr_full.generate_random_ride_loop(min_length=40,  max_length=100, seed=1)
_loop2 = _mgr_full.generate_random_ride_loop(min_length=40, max_length=100, seed=7)

route1 = JeepneyRoute("R1-Poblacion-Loop",  _loop1[:-1])
route2 = JeepneyRoute("R2-Hinaplanon-Loop", _loop2[:-1])

print(route1)
print(route2)


[TravelGraphManager] full ride graph | 12,855 nodes | 47,463 edges
  Edge types : ['alight', 'direct', 'end_walk', 'ride', 'start_walk', 'transfer', 'wait']
  Weights    — walk: 0.0142, ride: 0.0071, wait: 8.5, transfer: 14.2
JeepneyRoute(id='R1-Poblacion-Loop', stops=89)
JeepneyRoute(id='R2-Hinaplanon-Loop', stops=123)


In [ ]:

# ── Step 3: build the route-restricted manager ────────────────────────────────
# Replace placeholder β values with your calibrated survey estimates.
mgr = TravelGraphManager(
    edges_csv   = TRAVEL_GRAPH_EDGES_CSV,
    nodes_csv   = TRAVEL_GRAPH_NODES_CSV,   # required for visualise_path()
    routes      = [route1, route2],
    walk_wt     = 2,
    ride_wt     = 1,
    wait_wt     = 1,
    transfer_wt = 2,
)
print(mgr)


[TravelGraphManager] 2 routes | 9,510 nodes | 28,866 edges
  Edge types : ['alight', 'direct', 'end_walk', 'ride', 'start_walk', 'transfer', 'wait']
  Weights    — walk: 0.0142, ride: 0.0071, wait: 8.5, transfer: 14.2
TravelGraphManager(2 routes, nodes=9,510, edges=28,866)


In [105]:

# ── Example: shortest path with the restricted route set ─────────────────────
import random as _random

_sw_nodes = mgr.edges[mgr.edges["u"].str.startswith("sw_")]["u"].unique()
_ew_nodes = mgr.edges[mgr.edges["v"].str.startswith("ew_")]["v"].unique()
_rng      = _random.Random()
_origin   = _rng.choice(_sw_nodes)
_dest     = _rng.choice(_ew_nodes)

print(f"Origin : {_origin}")
print(f"Dest   : {_dest}")

try:
    _path_edges = mgr.calculate_shortest_path(_origin, _dest)
    _etypes = [mgr.get_edge(e).edge_type for e in _path_edges if mgr.get_edge(e)]
    print(f"Path   : {len(_path_edges)} edges")
    print(f"Types  : {' → '.join(_etypes)}")
except nx.NetworkXNoPath:
    print(
        "No path found with the current routes — the origin/dest may not be "
        "reachable under routes R1 and R2. Try different nodes or more routes."
    )


Origin : sw_1553790713
Dest   : ew_10879399389
Path   : 64 edges
Types  : direct → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk → end_walk


In [106]:

# ── Example: visualise the journey ───────────────────────────────────────────
# Requires a successful path_edges result from the cell above.
_viz_path = OUTPUT_DIR / "journey_map.html"

try:
    mgr.visualize_path(
        path_edges  = _path_edges,
        output_html = _viz_path,
        title       = "Iligan Jeepney Journey",
    )
    show_html_preview(_viz_path)
except (NameError, ValueError, RuntimeError) as e:
    print(f"Visualisation skipped: {e}")


Saved journey map → C:\Users\lifei\OneDrive\Desktop\Thesis\Thesis Repository\Thesis\notebooks\outputs\journey_map.html
